Importing necessary libraries and the data

In [2]:
from IPython.display import display

from config import DATA_DIR
import pandas as pd
from datetime import datetime

df = pd.read_excel(DATA_DIR / "dataset_exercise.xlsx")
df.head()



,start_alarm,end_alarm,alarm_id,machine_state,time_window
0,2025-06-23 03:08:36.666,2025-06-23 03:08:38.663,156700113,Failure,1
1,2025-06-23 03:08:36.666,2025-06-23 03:08:38.663,156700114,Failure,1
2,2025-06-23 03:08:42.154,2025-06-23 03:10:12.667,156701901,Failure,1
3,2025-06-23 03:08:42.669,2025-06-23 03:10:12.667,156702002,Failure,1
4,2025-06-23 03:08:42.669,2025-06-23 03:10:12.667,156701902,Failure,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4307 entries, 0 to 4306
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   start_alarm    4307 non-null   datetime64[us]
 1   end_alarm      4302 non-null   datetime64[us]
 2   alarm_id       4307 non-null   int64         
 3   machine_state  4307 non-null   str           
 4   time_window    4307 non-null   int64         
dtypes: datetime64[us](2), int64(2), str(1)
memory usage: 168.4 KB


Check to make sure the alarms are in order

In [4]:
if df["start_alarm"].index.is_monotonic_increasing != True:
    # not necessary as the start_alarm values are already in ascending order
    df = df.sort_values("start_alarm")
else:
    print("The start_alarm column in the dataframe is monotonic increasing\n")

The start_alarm column in the dataframe is monotonic increasing



Calculating the alarm durations (might be better to do it later so we have less durations to calculate, unnecessary to calculate it for all of the alarms if we don't use them)

Modifying the machine state to have binary values

In [5]:
window_df = df.groupby("time_window").agg(
    machine_state=("machine_state", "first")#,
    #start_time=("start_alarm", "min")
    ).reset_index()

display(window_df.head())
window_df.info()
#encoding_dict= {"machine_state": {"Failure": 1, "Running": 0}}

#window_df.replace(encoding_dict, inplace=True)
#window_df["machine_state"] = window_df["machine_state"].astype(int)
window_df["machine_state"] = (
    window_df["machine_state"]
    .map({
        "Failure": 1,
        "Running": 0
    })
)

display(window_df.head())
window_df.info()

,time_window,machine_state
0,1,Failure
1,2,Failure
2,4,Failure
3,5,Running
4,6,Running


<class 'pandas.DataFrame'>
RangeIndex: 329 entries, 0 to 328
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   time_window    329 non-null    int64
 1   machine_state  329 non-null    str  
dtypes: int64(1), str(1)
memory usage: 5.3 KB


,time_window,machine_state
0,1,1
1,2,1
2,4,1
3,5,0
4,6,0


<class 'pandas.DataFrame'>
RangeIndex: 329 entries, 0 to 328
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   time_window    329 non-null    int64
 1   machine_state  329 non-null    int64
dtypes: int64(2)
memory usage: 5.3 KB


Adding next state and failure_next (one if the machine fails in the next timeframe) columns

In [6]:
window_df["next_state"] = (
    window_df["machine_state"]
    .shift(-1)
    .fillna(0)
    .astype(int)
)


display(window_df.head())
window_df.info()
window_df["failure_next"] = (
    (window_df["machine_state"] == 0) &
    (window_df["next_state"] == 1)
).astype(int)

display(window_df.head(10))
window_df.info()

,time_window,machine_state,next_state
0,1,1,1
1,2,1,1
2,4,1,0
3,5,0,0
4,6,0,0


<class 'pandas.DataFrame'>
RangeIndex: 329 entries, 0 to 328
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   time_window    329 non-null    int64
 1   machine_state  329 non-null    int64
 2   next_state     329 non-null    int64
dtypes: int64(3)
memory usage: 7.8 KB


,time_window,machine_state,next_state,failure_next
0,1,1,1,0
1,2,1,1,0
2,4,1,0,0
3,5,0,0,0
4,6,0,0,0
5,8,0,1,1
6,20,1,1,0
7,21,1,0,0
8,22,0,0,0
9,23,0,0,0


<class 'pandas.DataFrame'>
RangeIndex: 329 entries, 0 to 328
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   time_window    329 non-null    int64
 1   machine_state  329 non-null    int64
 2   next_state     329 non-null    int64
 3   failure_next   329 non-null    int64
dtypes: int64(4)
memory usage: 10.4 KB


Extracting every instance of the machine failing

In [7]:
failure_windows = window_df.loc[
    window_df["failure_next"] == 1,
    "time_window"
]

display(failure_windows.head())

5      8
11    26
14    32
17    42
19    47
Name: time_window, dtype: int64

Creating a DataFrame which includes only the time_windows of the original df DataFrame, that are followed by failure

In [8]:
df_failure_windows = df[df["time_window"].isin(failure_windows)]

display(df_failure_windows.head())

,start_alarm,end_alarm,alarm_id,machine_state,time_window
84,2025-06-23 05:28:39.536,2025-06-23 05:29:29.539,156700612,Running,8
85,2025-06-23 05:29:33.542,2025-06-23 05:29:35.547,156700113,Running,8
86,2025-06-23 05:29:33.542,2025-06-23 05:29:35.547,156700114,Running,8
87,2025-06-23 05:29:39.044,2025-06-23 05:30:11.042,156701909,Running,8
88,2025-06-23 05:29:39.044,2025-06-23 05:30:11.042,156701901,Running,8


Counting how many times each alarm occurred in these time_windows.

In [9]:
alarm_counts = (
    df_failure_windows["alarm_id"]
    .value_counts()
).reset_index()

display(alarm_counts.head(10))

#most influential alarms
a1 = alarm_counts["alarm_id"].iloc[0]
a2 = alarm_counts["alarm_id"].iloc[1]
a3 = alarm_counts["alarm_id"].iloc[3]


selected_alarms = [a1, a2, a3]
print("The selected alarms are:")
print(selected_alarms)

,alarm_id,count
0,156701909,78
1,156701901,78
2,156701902,77
3,156700113,60
4,156700114,60
5,156702002,15
6,156701900,15
7,156700004,10
8,156600208,10
9,156700400,8


The selected alarms are:
[np.int64(156701909), np.int64(156701901), np.int64(156700113)]


Extracting the number of times each selected alarm has occurred, and replacing the time_windows in which none of them were present.

In [10]:
alarm_features = (
    df[df["alarm_id"].isin(selected_alarms)]
      .groupby(["time_window", "alarm_id"])
      .size()
      .unstack(fill_value=0)
)

display(alarm_features.head())
alarm_features.info()

#replacing the windows that didn't include any of the alarms
alarm_features = alarm_features.reindex(df["time_window"].unique(), fill_value=0)

alarm_features = alarm_features.rename(columns={
    a1: "A1_count",
    a2: "A2_count",
    a3: "A3_count"
})

alarm_features = alarm_features.reset_index()

display(alarm_features.tail())
alarm_features.info()

alarm_id,156700113,156701901,156701909
time_window,,,
1,3,3,3
4,3,3,3
5,4,3,3
6,1,3,3
8,2,2,2


<class 'pandas.DataFrame'>
Index: 262 entries, 1 to 837
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   156700113  262 non-null    int64
 1   156701901  262 non-null    int64
 2   156701909  262 non-null    int64
dtypes: int64(3)
memory usage: 8.2 KB


alarm_id,time_window,A3_count,A2_count,A1_count
324,831,3,3,3
325,833,1,1,1
326,834,1,1,1
327,836,1,2,2
328,837,0,1,1


<class 'pandas.DataFrame'>
RangeIndex: 329 entries, 0 to 328
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   time_window  329 non-null    int64
 1   A3_count     329 non-null    int64
 2   A2_count     329 non-null    int64
 3   A1_count     329 non-null    int64
dtypes: int64(4)
memory usage: 10.4 KB


In [11]:
def discretize_counts(x):
    if x == 0:
        return 0
    elif x <= 2:
        return 1
    elif x <= 5:
        return 2
    else:
        return 3

In [12]:
alarm_features_discrete = alarm_features.set_index("time_window").apply(
    lambda col: col.map(discretize_counts)
).reset_index()
display(alarm_features_discrete)
alarm_features_discrete.info()

alarm_id,time_window,A3_count,A2_count,A1_count
0,1,2,2,2
1,2,0,0,0
2,4,2,2,2
3,5,2,2,2
4,6,1,2,2
...,...,...,...,...
324,831,2,2,2
325,833,1,1,1
326,834,1,1,1
327,836,1,1,1


<class 'pandas.DataFrame'>
RangeIndex: 329 entries, 0 to 328
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   time_window  329 non-null    int64
 1   A3_count     329 non-null    int64
 2   A2_count     329 non-null    int64
 3   A1_count     329 non-null    int64
dtypes: int64(4)
memory usage: 10.4 KB


In [13]:
print(alarm_features_discrete.max(axis=0))

alarm_id
time_window    837
A3_count         3
A2_count         3
A1_count         3
dtype: int64


In [14]:
duration_features = (
    df[df["alarm_id"].isin(selected_alarms)]
      .assign(
          alarm_duration=lambda x:
              (x["end_alarm"] - x["start_alarm"]).dt.total_seconds()
      )
      .groupby(["time_window", "alarm_id"])["alarm_duration"]
      .sum()
      .unstack(fill_value=0)
)

duration_features = duration_features.reindex(
    df["time_window"].unique(),
    fill_value=0
)

duration_features = duration_features.rename(columns={
    a1: "A1_duration",
    a2: "A2_duration",
    a3: "A3_duration"
})

duration_features = duration_features.reset_index()
duration_features

alarm_id,time_window,A3_duration,A2_duration,A1_duration
0,1,6.002,3235.019,3233.522
1,2,0.000,0.000,0.000
2,4,5.979,1607.941,1607.449
3,5,8.499,36.990,36.990
4,6,2.006,171.495,169.976
...,...,...,...,...
324,831,6.027,96.499,95.018
325,833,2.015,31.006,31.006
326,834,1.980,10.014,9.502
327,836,2.004,540.980,539.987


In [15]:
def discretize_durations(x):
    if x == 0:
        return 0
    elif x <= 30:
        return 1
    elif x <= 300:
        return 2
    else:
        return 3

In [16]:
duration_features_discrete = duration_features.apply(lambda col: col.map(discretize_durations))
display(duration_features_discrete)
duration_features_discrete.info()

alarm_id,time_window,A3_duration,A2_duration,A1_duration
0,1,1,3,3
1,1,0,0,0
2,1,1,3,3
3,1,1,2,2
4,1,1,2,2
...,...,...,...,...
324,3,1,2,2
325,3,1,2,2
326,3,1,1,1
327,3,1,3,3


<class 'pandas.DataFrame'>
RangeIndex: 329 entries, 0 to 328
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   time_window  329 non-null    int64
 1   A3_duration  329 non-null    int64
 2   A2_duration  329 non-null    int64
 3   A1_duration  329 non-null    int64
dtypes: int64(4)
memory usage: 10.4 KB


In [17]:
print(duration_features_discrete.max(axis=0))

alarm_id
time_window    3
A3_duration    1
A2_duration    3
A1_duration    3
dtype: int64


In [18]:
df2 = pd.DataFrame(columns=["time_window", "A1_count", "A1_duration", "A2_count",
                            "A2_duration", "A3_count", "A3_duration", "State_t", "State_t+1" ])

df2["time_window"]=window_df["time_window"]
df2["State_t"]=window_df["machine_state"]
df2["State_t+1"]=window_df["next_state"]
df2["A1_count"]=alarm_features_discrete["A1_count"]
df2["A2_count"]=alarm_features_discrete["A2_count"]
df2["A3_count"]=alarm_features_discrete["A3_count"]
df2["A1_duration"]=duration_features_discrete["A1_duration"]
df2["A2_duration"]=duration_features_discrete["A2_duration"]
df2["A3_duration"]=duration_features_discrete["A3_duration"]
df2

,time_window,A1_count,A1_duration,A2_count,A2_duration,A3_count,A3_duration,State_t,State_t+1
0,1,2,3,2,3,2,1,1,1
1,2,0,0,0,0,0,0,1,1
2,4,2,3,2,3,2,1,1,0
3,5,2,2,2,2,2,1,0,0
4,6,2,2,2,2,1,1,0,0
...,...,...,...,...,...,...,...,...,...
324,831,2,2,2,2,2,1,0,0
325,833,1,2,1,2,1,1,0,0
326,834,1,1,1,1,1,1,0,1
327,836,1,3,1,3,1,1,1,1


Define the Prediction Target

In [19]:
target = df2["State_t+1"]
target

0      1
1      1
2      0
3      0
4      0
      ..
324    0
325    0
326    1
327    1
328    0
Name: State_t+1, Length: 329, dtype: int64

Define observation window/feature set

In [20]:
df_dbn = df2.copy().drop("time_window", axis=1)
df_dbn

,A1_count,A1_duration,A2_count,A2_duration,A3_count,A3_duration,State_t,State_t+1
0,2,3,2,3,2,1,1,1
1,0,0,0,0,0,0,1,1
2,2,3,2,3,2,1,1,0
3,2,2,2,2,2,1,0,0
4,2,2,2,2,1,1,0,0
...,...,...,...,...,...,...,...,...
324,2,2,2,2,2,1,0,0
325,1,2,1,2,1,1,0,0
326,1,1,1,1,1,1,0,1
327,1,3,1,3,1,1,1,1


In [21]:
# rename to plain variable names, then assign proper (var, time_slice) tuples

df_dbn = df_dbn.rename(columns={

    "State_t": ("State", 0),

    "State_t+1": ("State", 1),

    "A1_count": ("A1_count", 0),

    "A1_duration": ("A1_duration", 0),

    "A2_count": ("A2_count", 0),

    "A2_duration": ("A2_duration", 0),

    "A3_count": ("A3_count", 0),

    "A3_duration": ("A3_duration", 0),

})

df_dbn.columns = pd.MultiIndex.from_tuples(df_dbn.columns)

In [22]:
for var in ["A1_count", "A1_duration", "A2_count", "A2_duration", "A3_count", "A3_duration"]:
    df_dbn[(var, 1)] = df_dbn[(var, 0)]

Creating the Dynamic Bayesian Network

In [23]:
from pgmpy.models import DynamicBayesianNetwork as DBN

dbn = DBN()

In [24]:
dbn.add_edges_from([
    (("A1_count", 0), ("A1_count", 1)),
    (("A1_duration", 0), ("A1_duration", 1)),
    (("A2_count", 0), ("A2_count", 1)),
    (("A2_duration", 0), ("A2_duration", 1)),
    (("A3_count", 0), ("A3_count", 1)),
    (("A3_duration", 0), ("A3_duration", 1)),

    (("A1_count", 0), ("State", 1)),
    (("A1_duration", 0), ("State", 1)),
    (("A2_count", 0), ("State", 1)),
    (("A2_duration", 0), ("State", 1)),
    (("A3_count", 0), ("State", 1)),
    (("A3_duration", 0), ("State", 1)),
    (("State", 0), ("State", 1)),
])

In [25]:
list(dbn.nodes())

[<DynamicNode(A1_count, 0) at 0x7b39237fc6e0>,
 <DynamicNode(A1_count, 1) at 0x7b3872c79f90>,
 <DynamicNode(A1_duration, 0) at 0x7b3a23307a80>,
 <DynamicNode(A1_duration, 1) at 0x7b3872c6a9e0>,
 <DynamicNode(A2_count, 0) at 0x7b3872ae1590>,
 <DynamicNode(A2_count, 1) at 0x7b3872ae16a0>,
 <DynamicNode(A2_duration, 0) at 0x7b3872adce50>,
 <DynamicNode(A2_duration, 1) at 0x7b38c95059a0>,
 <DynamicNode(A3_count, 0) at 0x7b3872bf2dd0>,
 <DynamicNode(A3_count, 1) at 0x7b3872bf3230>,
 <DynamicNode(A3_duration, 0) at 0x7b3a234e7a10>,
 <DynamicNode(A3_duration, 1) at 0x7b3a23238b90>,
 <DynamicNode(State, 1) at 0x7b3872b0d810>,
 <DynamicNode(State, 0) at 0x7b3872b0d950>]

In [26]:
dbn.fit(df_dbn) #max likelihood estimator might lead to 0 probabilities

In [27]:
#print(list(dbn.nodes()))
#print(dbn.edges())
#print([cpd.variable for cpd in dbn.get_cpds()])

In [68]:
from pgmpy.inference import DBNInference

dbn_infer = DBNInference(dbn)

evidence = {
    ("A1_count", 0): 2, #0-3 None, Low, Medium, High
    ("A1_duration", 0): 1, #0-3 None, Short, Medium, Long
    ("A2_count", 0): 2,
    ("A2_duration", 0): 2,
    ("A3_count", 0): 2,
    ("A3_duration", 0): 1, #0-1
    ("State", 0): 0, #0: Running, 1: Failure
}

In [70]:
result = dbn_infer.query(variables=[("State", 1)], evidence=evidence)
print(result[("State", 1)])

+-----------------+---------------------+
| ('State', 1)    |   phi(('State', 1)) |
+=================+=====================+
| ('State', 1)(0) |              0.5000 |
+-----------------+---------------------+
| ('State', 1)(1) |              0.5000 |
+-----------------+---------------------+
